# 4. Hyperparameter Tuning

This notebook tunes the candidate machine learning classifiers on the SMOTENC-resampled training data using **GridSearchCV** with **Stratified K-Fold**.

To prevent extremely long training times on the 1.13M row dataset, we use an optimized, lightweight parameter search space. Tuning is evaluated using F1 score, and the best tuned models are saved for downstream use.

## 1. Imports & Setup

In [1]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print('Imports complete.')

Imports complete.


## 2. Load Prepared Dataset

In [2]:
BASE_DIR = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'artifacts').exists() or (candidate / 'requirements.txt').exists():
        BASE_DIR = candidate
        break
if BASE_DIR is None:
    BASE_DIR = Path.cwd()

ARTIFACT_DIR = BASE_DIR / 'artifacts' / 'data'
MODEL_DIR    = BASE_DIR / 'artifacts' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_train.npz')['X_train']
X_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_test.npz')['X_test']
y_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_train.npz')['y_train']
y_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_test.npz')['y_test']

with open(ARTIFACT_DIR / 'features.json', 'r') as f:
    feature_names = json.load(f)

with open(ARTIFACT_DIR / 'threshold.json', 'r') as f:
    OPTIMAL_THRESHOLD = json.load(f)['optimal_threshold']

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)

print(f'X_train shape: {X_train.shape}  (fraud count: {int(y_train.sum()):,})')
print(f'X_test shape : {X_test.shape}   (fraud count: {int(y_test.sum()):,})')
print(f'Optimal threshold: {OPTIMAL_THRESHOLD}')

X_train shape: (1134468, 29)  (fraud count: 103,133)
X_test shape : (259335, 29)   (fraud count: 1,501)
Optimal threshold: 0.7


## 3. Define Candidate Models & Parameter Grids

In [3]:
models = {
    'Logistic Regression': LogisticRegression(random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(random_state=SEED, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=SEED, eval_metric='logloss', n_jobs=-1)
}

# Optimized grids for quick training
param_grids = {
    'Logistic Regression': {
        'max_iter': [1000]
    },
    'Decision Tree': {
        'max_depth': [12, 16],
        'criterion': ['entropy']
    },
    'Random Forest': {
        'n_estimators': [50],
        'max_depth': [8, 12]
    },
    'XGBoost': {
        'n_estimators': [100],
        'max_depth': [6, 8],
        'learning_rate': [0.1, 0.2]
    }
}

cv = StratifiedKFold(n_splits=6, random_state=SEED, shuffle=True)
print('Models and parameter grids defined.')

Models and parameter grids defined.


## 4. Run Hyperparameter Tuning via GridSearchCV

In [4]:
grid_search_results = {}
best_params = {}

for model_name, model in models.items():
    print(f'\n--- Tuning {model_name} ---')
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        cv=cv,
        scoring='f1',
        verbose=1,
        return_train_score=False
    )
    
    grid_search.fit(X_train_df, y_train)
    grid_search_results[model_name] = grid_search
    best_params[model_name] = grid_search.best_params_
    
    print(f'Best parameters: {grid_search.best_params_}')
    print(f'Best CV F1-Score: {grid_search.best_score_:.4f}')


--- Tuning Logistic Regression ---
Fitting 6 folds for each of 1 candidates, totalling 6 fits
Best parameters: {'max_iter': 1000}
Best CV F1-Score: 0.7893

--- Tuning Decision Tree ---
Fitting 6 folds for each of 2 candidates, totalling 12 fits
Best parameters: {'criterion': 'entropy', 'max_depth': 16}
Best CV F1-Score: 0.9618

--- Tuning Random Forest ---
Fitting 6 folds for each of 2 candidates, totalling 12 fits
Best parameters: {'max_depth': 12, 'n_estimators': 50}
Best CV F1-Score: 0.9273

--- Tuning XGBoost ---
Fitting 6 folds for each of 4 candidates, totalling 24 fits
Best parameters: {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 100}
Best CV F1-Score: 0.9825


## 5. Evaluate Tuned Models on Test Split

Evaluate best tuned estimators on the untouched test split using the optimal threshold of **0.70**.

In [5]:
evaluation_results = []

for model_name, grid_search in grid_search_results.items():
    best_estimator = grid_search.best_estimator_
    
    # Predict probabilities
    y_proba = best_estimator.predict_proba(X_test_df)[:, 1]
    y_pred = (y_proba >= OPTIMAL_THRESHOLD).astype(int)
    
    f1 = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    evaluation_results.append({
        'Model': model_name,
        'Test Accuracy': acc,
        'Test Precision': prec,
        'Test Recall': rec,
        'Test F1-Score': f1,
        'Test ROC-AUC': auc
    })

eval_df = pd.DataFrame(evaluation_results).set_index('Model')
print('\n=== Tuned Model Comparison Summary (threshold = 0.70) ===')
print(eval_df.round(4).to_string())


=== Tuned Model Comparison Summary (threshold = 0.70) ===
                     Test Accuracy  Test Precision  Test Recall  Test F1-Score  Test ROC-AUC
Model                                                                                       
Logistic Regression         0.9939          0.4695       0.4211         0.4440        0.9257
Decision Tree               0.9960          0.6146       0.8414         0.7103        0.9505
Random Forest               0.9976          0.8295       0.7355         0.7797        0.9937
XGBoost                     0.9978          0.8211       0.7981         0.8095        0.9977


## 6. Persist Tuned Models & Hyperparameters

In [6]:
# Save parameters to a JSON file
params_save_path = MODEL_DIR / 'best_tuned_params.json'
with open(params_save_path, 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'Saved best parameters to: {params_save_path}')

# Save the best estimators
for model_name, grid_search in grid_search_results.items():
    sanitized_name = model_name.lower().replace(' ', '_')
    model_save_path = MODEL_DIR / f'{sanitized_name}_tuned_model.pkl'
    joblib.dump(grid_search.best_estimator_, model_save_path)
    print(f'Saved tuned {model_name} model to: {model_save_path}')

Saved best parameters to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\best_tuned_params.json
Saved tuned Logistic Regression model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\logistic_regression_tuned_model.pkl
Saved tuned Decision Tree model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\decision_tree_tuned_model.pkl
Saved tuned Random Forest model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\random_forest_tuned_model.pkl
Saved tuned XGBoost model to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\xgboost_tuned_model.pkl
